### Agentic RAG with LangGraph

In [ ]:
import os
from typing import TypedDict, List

from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=os.getenv("API_KEY"),
    temperature=0
)

# Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=os.getenv("API_KEY")
)

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.0, client=<google.genai.client.Client object at 0x000001B6136EDA90>, default_metadata=(), model_kwargs={})

### State defination

In [22]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [25]:
from langchain_community.vectorstores import FAISS

# sample documets and vector store
sample_text = [
"The quick brown fox jumps over the lazy dog."
"This is a sample document for testing the retrieval"
"Rag is a powerful technique that combines the strengths of retrieval-based methods and generative models. "
"It allows for more accurate and contextually relevant responses by leveraging external knowledge sources.",
"LangGraph is a framework that enables the creation of complex workflows and pipelines for natural language processing tasks. "
"It provides a flexible and modular approach to building language models and applications.",
]

documets = [Document(page_content=text) for text in sample_text]

# create vector store
vectorstore = FAISS.from_documents(documets, embeddings)
retreiver = vectorstore.as_retriever(search_kwargs={"k": 2})


### Agentic Function

In [26]:
def decide_retrieval(state: AgentState) -> AgentState:
    """
    Decide whether to perform retrieval based on the question.
    """
    question = state["question"]

    # Simple heuristic: if the question contains certain keywords, we need retrieval
    retrieval_keywords = ["what", "who", "where", "when", "how", "why", "tell me about", "explain", "define", "describe", "give me information on"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval}

In [27]:
def retrieve_documents(state: AgentState) -> AgentState:
    """
    Retrieve documents based on the question.
    """
    question = state["question"]
    documents = retreiver.invoke(question)
    return {**state, "documents": documents}

In [29]:
def generate_answer(state: AgentState) -> AgentState:
    """
    Generate an answer based on the question and retrieved documents.
    """

    question = state["question"]
    documents = state.get("documents", [])

    if documents:
        context = "\n\n".join(
            [doc.page_content for doc in documents]
        )

        prompt = f"""
                Answer the following question based only on the provided context.

                Context:
                {context}

                Question:
                {question}

                Answer:
                """

    else:
        prompt = f"""
                Answer the following question:

                Question:
                {question}

                Answer:
                """

    response = llm.invoke(prompt)
    return {
        **state,"answer": response.content
    }

### Conditional Logic

In [30]:
def should_retrieve(state: AgentState) -> bool:
    """
    Determine if retrieval is needed based on the state.
    """
    if state["needs_retrieval"]:
        return "retrieve"
    else:
        return "generate"

### Build the Graph

In [36]:
# Create a state graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("decide", decide_retrieval)
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_answer)

# Set entry point
workflow.set_entry_point("decide")

# Conditional routing
workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)